# Universality / Rescaling Analysis

Tests whether different stimulus categories produce the same forgetting curve
shape up to a linear transformation.

**Analyses:**
1. Load multi-ISI data for multiple stimulus categories
2. Overlay raw forgetting curves
3. Linear rescaling across categories
4. Model comparison: linear, logarithmic, power-law fits to average curve
5. Piecewise fitting (small vs large ISI regions)

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt

from utls.loading import load_results_with_exclusion_no_dropping
from utls.dprime import recompute_dprime_by_isi_per_subject

from utls.data_loading import load_multi_isi, HR_TASK_NAMES, make_save_dir
from utls.human_analysis import run_analysis
from utls.scaling import (
    linear_scale, fit_forgetting_models, fit_piecewise_models,
    plot_universality, plot_model_fits, plot_piecewise_fits,
)

## Configuration

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────
TASKS = ["env-sounds", "glob-music", "atexts"]
TASK_NAMES = [HR_TASK_NAMES[t] for t in TASKS]
REFERENCE_IDX = 0  # which task to use as reference for scaling (env-sounds)

GRID = np.array([0, 1, 2, 4, 8, 16, 32, 64])  # shared ISI grid
TRANSFER_POINT = 8  # boundary for piecewise fitting

FIGURES_BASE = "/om2/user/bjmedina/auditory-memory/memory/figures/human-results"
save_dir = make_save_dir(FIGURES_BASE, "all", sub="multi-isi")
print(f"Figures saved to: {save_dir}")

## 1. Load Data for All Categories

In [ ]:
analyses = {}
for task, name in zip(TASKS, TASK_NAMES):
    data = load_multi_isi(
        task=task,
        load_fn=load_results_with_exclusion_no_dropping,
        min_dprime=2,
        min_trials=120,
    )
    x = recompute_dprime_by_isi_per_subject(data['exps'])
    out = run_analysis(x)
    analyses[name] = out
    print(f"{name}: N={out['N']}, d'(0)={out['dprime'][0]:.2f}, d'(64)={out['dprime'][-1]:.2f}")

## 2. Interpolate onto Shared Grid

In [ ]:
curves = {}
for name, out in analyses.items():
    curves[name] = np.interp(GRID, out['isis'], out['dprime'])

curve_list = list(curves.values())
name_list = list(curves.keys())

# Average curve
avg_curve = np.mean(curve_list, axis=0)
print(f"Average curve: {avg_curve}")

## 3. Linear Rescaling

Scale each non-reference curve to match the reference via y = a*x + b.

In [ ]:
ref_name = name_list[REFERENCE_IDX]
ref_curve = curve_list[REFERENCE_IDX]

scale_results = []
scaled_curves = []
for i, (name, curve) in enumerate(zip(name_list, curve_list)):
    if i == REFERENCE_IDX:
        # Reference maps to itself
        scale_results.append({'a': 1.0, 'b': 0.0, 'scaled': curve, 'r': 1.0})
        scaled_curves.append(curve)
    else:
        info = linear_scale(curve, ref_curve)
        scale_results.append(info)
        scaled_curves.append(info['scaled'])
        print(f"{name} -> {ref_name}: a={info['a']:.3f}, b={info['b']:.3f}, r={info['r']:.3f}")

In [ ]:
plot_universality(
    GRID, curve_list, name_list,
    scaled_curves=scaled_curves,
    avg_curve=avg_curve,
    scale_info=scale_results,
    title_prefix=f"Universality: ",
    save_path=os.path.join(save_dir, "universality_scaled_curves_all_scales.png"),
)

## 4. Model Comparison on Average Curve

In [ ]:
models = fit_forgetting_models(GRID, avg_curve)

print("=== MODEL FIT COMPARISON (Average Curve) ===")
for name, res in models.items():
    print(f"  {name:12s} R² = {res['r2']:.3f}")

In [ ]:
plot_model_fits(
    GRID, avg_curve, models,
    title="Model Fits to Average Forgetting Curve",
    save_path=os.path.join(save_dir, "model_fit_comparison.png"),
)

## 5. Piecewise Fitting (Small vs Large ISI)

In [ ]:
pw = fit_piecewise_models(GRID, avg_curve, transfer_point=TRANSFER_POINT)

print("=== PIECEWISE FITS ===")
for region in ['small', 'large']:
    for model in ['log', 'power']:
        r2 = pw[region][model]['r2']
        print(f"  {region:5s} / {model:5s}: R² = {r2:.3f}")

In [ ]:
plot_piecewise_fits(
    GRID, avg_curve, pw,
    title=f"Piecewise Fits: ISI < {TRANSFER_POINT} vs ISI >= {TRANSFER_POINT}",
    save_path=os.path.join(save_dir, "piecewise_fits.png"),
)